# 📦 Notebook 1 — Data Loading & Cleaning
**Project**: Marine Ecosystem Health Classifier  
**Dataset**: CalCOFI (Scripps) — Bottle + Cast + Zooplankton  
**Output**: `data/cleaned_bottle.parquet`, `data/cleaned_cast.parquet`, `data/cleaned_zoo.parquet`

In [2]:
import re
import io
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.3f}'.format)
print('Libraries loaded ✅')

ModuleNotFoundError: No module named 'numpy'

## 1.1 — Load CalCOFI SQL Dump
Stream-parse the 400MB file without loading it fully into RAM.

In [3]:
SQL_PATH = '../data/CalCOFI_Database_194903-202105_sql_16October2023.zip'
ZOO_PATH = '../data/erdCalCOFIzoovol_7093_319a_f698.csv'

BOTTLE_COLS = [
    'Cst_Cnt','Btl_Cnt','Sta_ID','Depth_ID','Depthm','T_degC',
    'Salnty','O2ml_L','STheta','O2Sat','Oxy_umol_Kg','BtlNum',
    'RecInd','T_prec','T_qual','S_prec','S_qual','P_qual','O_qual',
    'SThtaq','O2Satq','ChlorA','Chlqua','Phaeop','Phaqua',
    'PO4uM','PO4q','SiO3uM','SiO3qu','NO2uM','NO2q',
    'NO3uM','NO3q','NH3uM','NH3q',
    'C14As1','C14A1p','C14A1q','C14As2','C14A2p','C14A2q',
    'DarkAs','DarkAp','DarkAq','MeanAs','MeanAp','MeanAq',
    'IncTim','LightP','R_Depth','R_TEMP','R_Sal','R_DYNHT',
    'R_Nuts','R_Oxy_umol_Kg','DIC1','DIC2','TA1','TA2',
    'pH1','pH2','DIC_Quality_Comment'
]

CAST_COLS = [
    'Cst_Cnt','Cruise_ID','Cruise','Cruz_Sta','DbSta_ID','Cast_ID',
    'Sta_ID','Quarter','Sta_Code','Distance','Date','Year','Month',
    'Julian_Date','Julian_Day','Time','Lat_Dec','Lat_Deg','Lat_Min',
    'Lat_Hem','Lon_Dec','Lon_Deg','Lon_Min','Lon_Hem',
    'Rpt_Line','St_Line','Ac_Line','Rpt_Sta','St_Station','Ac_Sta',
    'Bottom_D','Cast_Secchi','Ship_Name','Ship_Code','Data_Type',
    'Order_Occ','Event_Num','Cruz_Leg','Orig_Sta_ID','Data_Or',
    'Cruz_Num','IntChl','IntC14','Inc_Str','Inc_End','PST_LAN',
    'Civil_T','TimeZone','Wave_Dir','Wave_Ht','Wave_Prd',
    'Wind_Dir','Wind_Spd','Barometer','Dry_T','Wet_T',
    'Wea','Cloud_Typ','Cloud_Amt','Visibility'
]

def _parse_sql_values(raw):
    vals, current, in_str = [], [], False
    for i, ch in enumerate(raw):
        if ch == "'" and not in_str: in_str = True
        elif ch == "'" and in_str:   in_str = False
        elif ch == ',' and not in_str:
            token = ''.join(current).strip()
            vals.append(None if token.upper() == 'NULL' else token.strip("'"))
            current = []; continue
        current.append(ch)
    token = ''.join(current).strip()
    vals.append(None if token.upper() == 'NULL' else token.strip("'"))
    return vals

def parse_table(zip_path, table_name, columns):
    ls  = subprocess.run(['unzip','-l',zip_path], capture_output=True, text=True)
    inner = [l.split()[-1] for l in ls.stdout.splitlines() if l.strip().endswith('.sql')][0]
    proc  = subprocess.Popen(['unzip','-p',zip_path,inner],
                              stdout=subprocess.PIPE, stderr=subprocess.DEVNULL)
    stream  = io.TextIOWrapper(proc.stdout, encoding='utf-8', errors='replace')
    pattern = re.compile(rf"INSERT INTO `{table_name}` \(\) VALUES \((.+)\);", re.I)
    rows = []
    for line in stream:
        m = pattern.match(line.strip())
        if m:
            vals = _parse_sql_values(m.group(1))
            if len(vals) == len(columns):
                rows.append(vals)
    return pd.DataFrame(rows, columns=columns)

print('Parsing Bottle table — this takes ~2 min...')
bottle = parse_table(SQL_PATH, 'Bottle', BOTTLE_COLS)
print(f'Bottle: {bottle.shape}')

print('Parsing Cast table...')
cast = parse_table(SQL_PATH, 'Cast', CAST_COLS)
print(f'Cast: {cast.shape}')

Parsing Bottle table — this takes ~2 min...


IndexError: list index out of range

## 1.2 — Type Coercion + Quality Flag Filtering
CalCOFI uses `9` as a missing/bad data code in quality flag columns.

In [4]:
# ── Numeric coercion ──────────────────────────────────────────────────────
NUM_BOTTLE = ['Cst_Cnt','Depthm','T_degC','Salnty','O2ml_L','O2Sat',
              'Oxy_umol_Kg','STheta','ChlorA','Phaeop',
              'PO4uM','SiO3uM','NO2uM','NO3uM','NH3uM',
              'T_qual','S_qual','O_qual','T_prec','R_Depth','R_TEMP','R_DYNHT']

NUM_CAST   = ['Cst_Cnt','Year','Month','Julian_Day','Quarter',
              'Lat_Dec','Lon_Dec','St_Line','St_Station','IntChl','Bottom_D']

for col in NUM_BOTTLE:
    if col in bottle.columns:
        bottle[col] = pd.to_numeric(bottle[col], errors='coerce')

for col in NUM_CAST:
    if col in cast.columns:
        cast[col] = pd.to_numeric(cast[col], errors='coerce')

# ── Filter bad quality flags (9 = missing/bad in CalCOFI convention) ──────
# Keep rows where temp and oxygen quality are acceptable (not 9)
bottle_clean = bottle[
    (bottle['T_qual'].isna()  | (bottle['T_qual']  != 9)) &
    (bottle['O_qual'].isna()  | (bottle['O_qual']  != 9))
].copy()

print(f'Rows before quality filter: {len(bottle):,}')
print(f'Rows after  quality filter: {len(bottle_clean):,}')
print(f'Dropped: {len(bottle) - len(bottle_clean):,} ({(1 - len(bottle_clean)/len(bottle))*100:.1f}%)')

NameError: name 'bottle' is not defined

## 1.3 — Missingness Audit

In [5]:
KEY_COLS = ['T_degC','Salnty','O2ml_L','O2Sat','ChlorA','PO4uM','NO3uM']
miss = bottle_clean[KEY_COLS].isnull().mean().sort_values(ascending=False) * 100

fig, ax = plt.subplots(figsize=(8,4))
miss.plot(kind='barh', ax=ax, color='#3b82f6', edgecolor='none')
ax.set_xlabel('% Missing')
ax.set_title('Missingness in Key Bottle Columns')
ax.axvline(50, color='red', linestyle='--', alpha=0.5, label='50% threshold')
ax.legend()
plt.tight_layout()
plt.show()
print(miss.to_string())

NameError: name 'bottle_clean' is not defined

## 1.4 — Load & Clean Zooplankton CSV

In [6]:
zoo = pd.read_csv(ZOO_PATH, header=0)
# Drop units row (row 0 contains column unit strings)
zoo = zoo[pd.to_numeric(zoo['latitude'], errors='coerce').notna()].copy()
zoo = zoo.rename(columns={
    'latitude':      'lat',
    'longitude':     'lon',
    'time':          'datetime',
    'small_plankton':'small_biovol',
    'total_plankton':'total_biovol',
    'volume_sampled':'vol_sampled',
})
for col in ['lat','lon','small_biovol','total_biovol','vol_sampled','line','station','cruise']:
    zoo[col] = pd.to_numeric(zoo[col], errors='coerce')

zoo['year']  = (zoo['cruise'] // 100).astype('Int64')
zoo['month'] = (zoo['cruise'] %  100).astype('Int64')
zoo['datetime'] = pd.to_datetime(zoo['datetime'], errors='coerce', utc=True)

print(zoo.shape)
zoo.head()

NameError: name 'pd' is not defined

## 1.5 — Save Cleaned Parquet Files

In [7]:
bottle_clean.to_parquet('../data/cleaned_bottle.parquet', index=False)
cast.to_parquet('../data/cleaned_cast.parquet', index=False)
zoo.to_parquet('../data/cleaned_zoo.parquet', index=False)

print('Saved:')
print(f'  data/cleaned_bottle.parquet  — {len(bottle_clean):,} rows')
print(f'  data/cleaned_cast.parquet    — {len(cast):,} rows')
print(f'  data/cleaned_zoo.parquet     — {len(zoo):,} rows')

NameError: name 'bottle_clean' is not defined